# 08 -- Monte Carlo risk analysis

Paired script: `analysis/monte_carlo.py`. Seeded bootstrap resampling
(`analysis.resampling.seeded_bootstrap_indices`) of a trade P/L sequence to characterize
the distribution of final equity, max drawdown, and probability of hitting a ruin
threshold. Explicitly documents the i.i.d.-resampling simplification (does not preserve
real trade-sequence autocorrelation) in the script's own module docstring.

**Uses clearly-labelled SYNTHETIC P/L data.** Real-data run: PENDING.

In [ ]:
import sys
import tempfile
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from analysis.monte_carlo import run

In [ ]:
tmp_dir = Path(tempfile.mkdtemp(prefix="themba_montecarlo_demo_"))
pd.DataFrame({
    "trade_id": [f"t{i}" for i in range(6)],
    "profit": [30.0, -20.0, 15.0, -10.0, 25.0, -15.0],
}).to_csv(tmp_dir / "trades.csv", index=False)

In [ ]:
result = run(tmp_dir / "trades.csv", n_resamples=2000, seed=42, starting_equity=1000.0,
             ruin_threshold=500.0, output_json=tmp_dir / "mc.json", repo_path=PROJECT_ROOT.parents[1])

print(f"final_equity_mean       = {result.final_equity_mean:.2f}")
print(f"final_equity_95pct_CI   = [{result.final_equity_ci_lower:.2f}, {result.final_equity_ci_upper:.2f}]")
print(f"max_drawdown_pct_mean   = {result.max_drawdown_pct_mean:.4f}")
print(f"prob_ruin (<=500)       = {result.prob_ruin}")

# Determinism check: re-running with the identical seed reproduces the exact result.
result_repeat = run(tmp_dir / "trades.csv", n_resamples=2000, seed=42, starting_equity=1000.0,
                     ruin_threshold=500.0)
assert result == result_repeat

## Real-data run: PENDING

Requires real trade P/L history -- none exists yet.